In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

X_train = np.load('/path/to/training/data/X_train.npy') 
Y_train = np.load('/path/to/training/data/Y_train.npy') 

X_train = np.expand_dims(X_train, axis=-1) 

def build_resnet_3d(input_shape):
    inputs = keras.Input(shape=input_shape)

    x = layers.Conv3D(32, (3, 3, 3), padding="same", strides=(1, 1, 1),
                      kernel_regularizer=regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = resnet_3d_block(x, 32)
    x = resnet_3d_block(x, 64, strides=(2, 2, 2))  
    x = resnet_3d_block(x, 128, strides=(2, 2, 2))  
    x = resnet_3d_block(x, 256, strides=(2, 2, 2)) 

    x = layers.GlobalAveragePooling3D()(x)
    x = layers.Dropout(0.5)(x) 

    outputs = layers.Dense(1, activation="linear")(x)  

    model = keras.Model(inputs, outputs, name="ResNet3D_Regression")
    return model

input_shape = X_train.shape[1:]  
model = build_resnet_3d(input_shape)

model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
              loss="mean_squared_error",
              metrics=["mae"])

history = model.fit(X_train, Y_train, epochs=50, batch_size=16,
                    validation_split=0.2, verbose=1)

model.save("resnet3d_regression_model.h5")

print("✅ Training complete! Model saved.")


In [ ]:
from tensorflow.keras.utils import plot_model

# Print model summary
model.summary()

# Visualize the model architecture (optional)
plot_model(model, to_file="model_structure.png", show_shapes=True, show_layer_names=True)

# Print total trainable and non-trainable parameters
total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = total_params - trainable_params

print("\n✅ Model Statistics:")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Non-trainable Parameters: {non_trainable_params:,}")


In [ ]:
import numpy as np
import tensorflow as tf

# Load validation data
X_val = np.load("/path/to/validation/data/X_val.npy")  # Shape: (157, 26, 21, 3)
Y_val = np.load("/path/to/validation/data/Y_val.npy")  # Shape: (157,)

# Add depth dimension to match model input
X_val = np.expand_dims(X_val, axis=-1)  # New shape: (157, 26, 21, 3, 1)

# Load trained model (make sure you have saved your model as 'resnet3d_model.h5')
model = tf.keras.models.load_model("/path/to/trained/resnet3d_regression_model.h5")

# Make predictions
Y_pred = model.predict(X_val)  # Shape: (157,)

# Compute squared error (element-wise)
squared_errors = np.square(Y_pred.flatten() - Y_val)

# Compute mean squared error (MSE)
mse = np.mean(squared_errors)

# Print results
print("\n✅ Model Validation Results:")
print(f"Squared Errors (first 5 samples): {squared_errors[:5]}")
print(f"Mean Squared Error (MSE) on validation set: {mse:.6f}")


In [ ]:
import numpy as np
import tensorflow as tf

# Load training data
X_train = np.load("/path/to/training/data/X_train.npy")  # Shape: (529, 26, 21, 3)
Y_train = np.load("/path/to/training/data/Y_train.npy")  # Shape: (529,)

# Add depth dimension to match model input
X_train = np.expand_dims(X_train, axis=-1)  # New shape: (529, 26, 21, 3, 1)

# Load trained model (Ensure it's saved as 'resnet3d_model.h5')
model = tf.keras.models.load_model("/path/to/trained/resnet3d_regression_model.h5")

# Make predictions
Y_pred_train = model.predict(X_train)  # Shape: (529,)

# Compute squared error (element-wise)
squared_errors_train = np.square(Y_pred_train.flatten() - Y_train)

# Compute mean squared error (MSE)
mse_train = np.mean(squared_errors_train)

# Print results
print("\n✅ Model Training Set Evaluation Results:")
print(f"Squared Errors (first 5 samples): {squared_errors_train[:5]}")
print(f"Mean Squared Error (MSE) on training set: {mse_train:.6f}")


In [ ]:
import numpy as np
import tensorflow as tf

# Load test data
X_test = np.load("/path/to/testing/data/X_test.npy")  # Shape: (157, 26, 21, 3)
Y_test = np.load("/path/to/testing/data/Y_test.npy")  # Shape: (157,)

# Add depth dimension to match model input
X_test = np.expand_dims(X_test, axis=-1)  # New shape: (157, 26, 21, 3, 1)

# Load trained model (Ensure it's saved as 'resnet3d_model.h5')
model = tf.keras.models.load_model("/path/to/trained/resnet3d_regression_model.h5")

# Make predictions
Y_pred_test = model.predict(X_test)  # Shape: (157,)

# Compute squared errors
squared_errors_test = np.square(Y_pred_test.flatten() - Y_test)

# Compute Root Mean Squared Error (RMSE)
rmse_test = np.sqrt(np.mean(squared_errors_test))

# Print results
print("\n✅ Model Testing Set Evaluation Results:")
print(f"Root Mean Squared Error (RMSE) on test set: {rmse_test:.6f}")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import r2_score

# Load test data
X_test = np.load("/path/to/testing/data/X_test.npy")  # Shape: (157, 26, 21, 3)
Y_test = np.load("/path/to/testing/data/Y_test.npy")  # Shape: (157,)

# Add depth dimension to match model input
X_test = np.expand_dims(X_test, axis=-1)  # New shape: (157, 26, 21, 3, 1)

# Load trained model (Ensure it's saved as 'resnet3d_model.h5')
model = tf.keras.models.load_model("/path/to/trained/resnet3d_regression_model.h5")

# Make predictions
Y_pred = model.predict(X_test)  # Shape: (157,)

# Compute R² Score
r2 = r2_score(Y_test, Y_pred)

print(f"Coefficient of Determination (R²) on Testing Set: {r2:.4f}")
